**Eval Metrics**

In [ ]:


def clean_and_normalize(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.lower().strip()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return " ".join(text.split())

def calculate_final_metrics(run_records):
    total_questions = len(run_records)
    exact_matches = 0
    successful_retrievals = 0
    total_steps = 0
    premature_guess_failures = 0
    reasoning_failures = 0
    total_incorrect = 0

    for run in run_records:
        norm_agent = clean_and_normalize(run['agent_answer'])
        norm_gold = clean_and_normalize(run['gold_answer'])
        if(norm_agent == norm_gold):
            exact_matches += 1
        else:
            total_incorrect += 1
        gold_set = set(run['gold_supporting_titles'])
        retrieved_set = set(run['retrieved_titles'])
        hit_gold_doc = len(gold_set.intersection(retrieved_set)) > 0
        if hit_gold_doc:
            successful_retrievals += 1
        total_steps += run['steps_used']
        if not (norm_agent == norm_gold):
            if not hit_gold_doc:
                premature_guess_failures += 1
            else:
                reasoning_failures += 1

    metrics_summary = {
        "Exact Match (EM)": exact_matches / total_questions,
        "Retrieval Recall": successful_retrievals / total_questions,
        "Average Steps-to-Answer": total_steps / total_questions,
        "Premature-Stops (No Context)": premature_guess_failures / total_incorrect if total_incorrect > 0 else 0.0,
        "Reasoning Failures (Had Context)": reasoning_failures / total_incorrect if total_incorrect > 0 else 0.0
    }
    return metrics_summary

**Logging**

In [ ]:
def get_eval_records(collected_run_records):
    eval_records = []

    for record in collected_run_records:
        target_answer = record['gold_answer']

        gold_supporting_titles = []
        if isinstance(record['supporting_facts'], dict) and 'title' in record['supporting_facts']:
            gold_supporting_titles = record['supporting_facts']['title']
        elif isinstance(record['supporting_facts'], list):
            gold_supporting_titles = [fact['title'] for fact in record['supporting_facts'] if 'title' in fact]

        search_steps_used = 0
        retrieved_titles = []

        if record['trajectory']:
            for step in record['trajectory']:
                if "Search" in str(step.get("action", "")):
                    search_steps_used += 1
                    obs = step.get("observation", {})

                    if isinstance(obs, dict) and "title" in obs:
                        retrieved_titles.append(obs["title"])
                    elif isinstance(obs, str) and ":" in obs:
                        extracted_title = obs.split(":", 1)[0].replace("Title", "").strip()
                        retrieved_titles.append(extracted_title)

        eval_records.append({
            'question': record['question'],
            'agent_answer': record['agent_answer'],
            'gold_answer': target_answer,
            'retrieved_titles': retrieved_titles,
            'gold_supporting_titles': gold_supporting_titles,
            'steps_used': search_steps_used
        })

    return eval_records

In [ ]:
def init_logger(filename="/content/drive/MyDrive/Agentic RAG Team 1/agent_trajectories.txt"):

    with open(filename, "w", encoding="utf-8") as f:
        f.write("AGENTIC RAG SYSTEM EVALUATION    \n\n")


def log_eval_item(idx, question, trajectory, gold_ans, filename):
    with open(filename, 'a', encoding='utf-8') as f:
        f.write(f"\n--- QUESTION {idx + 1} ---\n")
        f.write(f"Prompt: {question}\n\n")

        if trajectory:
            for i, step in enumerate(trajectory):
                f.write(f"[Turn {i + 1}]\n")
                if "thought" in step:
                    f.write(f"  Thought: {step['thought']}\n")

                f.write(f"  Action: {step.get('action', '')}\n")

                if "query" in step:
                    f.write(f"  Query: {step['query']}\n")

                if "observation" in step:
                    obs_str = str(step['observation'])
                    f.write(f"  Observation: {obs_str[:300]}...\n")

                if "answer" in step:
                    f.write(f"  Final Extracted Answer: {step['answer']}\n")

        else:
            f.write("No trajectory recorded.\n")

        f.write(f"\nTarget Gold Answer: {gold_ans}\n\n")

In [ ]:
# data_path = '/content/drive/MyDrive/Agentic RAG Team 1/data.json'
# with open(data_path, 'r', encoding='utf-8') as f:
#     eval_subset = json.load(f)

# # eval_subset = full_dataset[:20]
# log_file_path = '/content/drive/MyDrive/Agentic RAG Team 1/agent_trajectories.txt'

# init_logger(log_file_path)

# collected_run_records = []
# print(f"Starting Agentic RAG Evaluation on {len(eval_subset)} questions...\n")

# #  Main Evaluation Loop
# for index, row in enumerate(eval_subset):
#     question_text = row["question"]
#     local_context = row["context"]
#     target_answer = row["answer"]
#     gold_supporting_titles = [fact["title"] for fact in row["supporting_facts"]]

#     # Generate global TF-IDF variables
#     context_sentences = [item["sentence"] for item in local_context]
#     tf_idf_vectorizer = TfidfVectorizer()
#     vectorized_matrix = tf_idf_vectorizer.fit_transform(context_sentences)

#     # Call the Agent
#     agent_answer, trajectory = run_agent(
#         question=question_text,
#         context=local_context
#     )

#     if agent_answer is None:
#         agent_answer = "Budget exhausted or invalid action."

#     # Parse trajectory safely
#     retrieved_titles = []
#     search_steps_used = 0

#     if trajectory:
#         for step in trajectory:
#             action_str = str(step.get("action", ""))
#             if "Search" in action_str:
#                 search_steps_used += 1
#                 obs = step.get("observation", {})

#                 if isinstance(obs, dict) and "title" in obs:
#                     retrieved_titles.append(obs["title"])
#                 elif isinstance(obs, str) and ":" in obs:
#                     extracted_title = obs.split(":", 1)[0].replace("Title", "").strip()
#                     retrieved_titles.append(extracted_title)

#     # Log and collect metrics
#     log_eval_item(index, question_text, trajectory, target_answer, log_file_path)

#     collected_run_records.append({
#         'question': question_text,
#         'agent_answer': agent_answer,
#         'gold_answer': target_answer,
#         'retrieved_titles': retrieved_titles,
#         'gold_supporting_titles': gold_supporting_titles,
#         'steps_used': search_steps_used
#     })

#     print(f"Finished Question {index + 1}/20")

#     time.sleep(12)

# #  Evaluate and Export
# final_metrics = calculate_final_metrics(collected_run_records)

# print("\nEVALUATION METRICS \n\n ")
# for metric_name, value in final_metrics.items():
#     print(f"-> {metric_name}: {value:.4f}")

# metrics_df = pd.DataFrame([final_metrics])
# metrics_df.to_csv("/content/drive/MyDrive/Agentic RAG Team 1/results_table.csv", index=False)
# print(f"\nSuccess, results saved to Drive: results_table.csv")

Starting Agentic RAG Evaluation on 20 questions...

Finished Question 1/20
Finished Question 2/20
Finished Question 3/20
Finished Question 4/20
Finished Question 5/20
Finished Question 6/20
Finished Question 7/20
Finished Question 8/20
Finished Question 9/20
Finished Question 10/20
Finished Question 11/20
Finished Question 12/20
Finished Question 13/20
Finished Question 14/20
Finished Question 15/20
Finished Question 16/20
Finished Question 17/20
Finished Question 18/20
Finished Question 19/20
Finished Question 20/20

EVALUATION METRICS 

 
-> Exact Match (EM): 0.4500
-> Retrieval Recall: 0.3000
-> Average Steps-to-Answer: 0.5000
-> Premature-Stops (No Context): 0.9091
-> Reasoning Failures (Had Context): 0.0909

Success, results saved to Drive: results_table.csv
